Cài đặt môi trường

In [1]:
import os, sys, json, re, shutil, subprocess, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

def find_tshark():
    """Tìm tshark: biến môi trường TSHARK -> PATH -> thư mục cài mặc định của Wireshark trên Windows."""
    cands = [os.environ.get('TSHARK'), shutil.which('tshark')]
    for base in (os.environ.get('ProgramFiles'), os.environ.get('ProgramFiles(x86)')):
        if base:
            cands.append(str(Path(base) / 'Wireshark' / 'tshark.exe'))
    for p in cands:
        if p and Path(p).is_file():
            return p
    raise FileNotFoundError('Không tìm thấy tshark. Hãy cài Wireshark (https://www.wireshark.org/download.html) '
                            'hoặc đặt biến môi trường TSHARK trỏ tới tshark.exe')

TSHARK = find_tshark()
print(TSHARK)
print(subprocess.run([TSHARK, '--version'], capture_output=True, text=True).stdout.splitlines()[0])

C:\Program Files\Wireshark\tshark.exe
TShark (Wireshark) 4.6.9 (v4.6.9-0-g2d548b197c75).


Cấu hình tham số và các biến

In [2]:
# Jupyter Lab chạy kernel với thư mục hiện hành = thư mục chứa notebook (ics-ai/notebooks)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_ROOT = Path(os.environ.get('CIC_MODBUS_ROOT', PROJECT_ROOT / 'Modbus-Dataset')).resolve()
WORK_DIR  = Path(os.environ.get('CIC_MODBUS_WORK', PROJECT_ROOT / 'Modbus-Work')).resolve()
LOCAL_CACHE = None         # None = đọc thẳng từ DATA_ROOT
TCP_DESEGMENT = False      # tắt ghép đoạn TCP
TCP_ANALYZE_SEQ = False    # tắt phân tích seq/ack (để không bỏ sót gói replay bị coi là retransmission)
N_WORKERS = min(4, os.cpu_count() or 1)   # tăng nếu máy nhiều lõi và nhiều RAM

# Capture point (đường dẫn tương đối so với DATA_ROOT) cho từng kịch bản
CAPTURE_POINTS = {
    'benign':            'benign/scada-hmi/scada-hmi-network-capture',
    'external':          'attack/external/network-wide',   
    'compromised_scada': 'attack/compromised-scada/scada-hmi/scada-hmi-network-capture',
    'compromised_ied':   'attack/compromised-ied/trust-scada-hmi/trust-scada-network-captures',
}
# Thư mục attack log của từng kịch bản tấn công (dùng ở notebook 02)
LOG_DIRS = {
    'external':          'attack/external/external-attacker/attacker logs',
    'compromised_scada': 'attack/compromised-scada/attack logs',
    'compromised_ied':   'attack/compromised-ied/attack logs',
}

# Số liệu bài báo để đối chiếu (Mục 3.1)
PAPER_FILES   = {'benign': 19, 'external': 2, 'compromised_scada': 17, 'compromised_ied': 20}
PAPER_PACKETS = {'benign': 4_101_373, 'external': 282_078, 'compromised_scada': 3_657_304, 'compromised_ied': 3_348_642}

assert DATA_ROOT.exists(), f'Không thấy DATA_ROOT: {DATA_ROOT}'
WORK_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / 'extracted').mkdir(exist_ok=True)

config = dict(data_root=str(DATA_ROOT), work_dir=str(WORK_DIR), tcp_desegment=TCP_DESEGMENT, tcp_analyze_seq=TCP_ANALYZE_SEQ,
              capture_points=CAPTURE_POINTS, log_dirs=LOG_DIRS,
              paper_files=PAPER_FILES, paper_packets=PAPER_PACKETS)
(WORK_DIR / 'config.json').write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding='utf-8')
print('Đã ghi', WORK_DIR / 'config.json')

Đã ghi C:\Users\cvson\Documents\Project\Anomaly-detection-in-ICS\ics-ai\Modbus-Work\config.json


Khảo sát toàn bộ thư mục pcap

In [3]:
def natural_key(p):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', Path(p).name)]

IP_NAMES = {'185.175.0.2': 'SecHMI', '185.175.0.3': 'HMI', '185.175.0.4': 'IED1A', '185.175.0.5': 'IED1B',
            '185.175.0.6': 'Agent', '185.175.0.7': 'Attacker', '185.175.0.8': 'IED4C'}

def probe_pcap(pcap, max_frames=100_000):
    """Đọc tối đa max_frames frame đầu, trả về (số frame Modbus, top cặp IP)."""
    cmd = [TSHARK, '-n', '-r', str(pcap), '-c', str(max_frames), '-Y', 'mbtcp',
           '-T', 'fields', '-e', 'ip.src', '-e', 'ip.dst']
    out = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8', errors='replace').stdout.splitlines()
    pairs = pd.Series([l for l in out if l.strip()], dtype=str)
    top = pairs.value_counts().head(3)
    top_s = ', '.join(f"{IP_NAMES.get(k.split()[0], k.split()[0])}→{IP_NAMES.get(k.split()[-1], k.split()[-1])}:{v}"
                      for k, v in top.items())
    return len(pairs), top_s

pcap_dirs = sorted({p.parent for p in DATA_ROOT.rglob('*.pcap')})
rows = []
for d in tqdm(pcap_dirs, desc='Khảo sát'):
    files = sorted(d.glob('*.pcap'), key=natural_key)
    n_mb, top = probe_pcap(files[0])
    rel = d.relative_to(DATA_ROOT).as_posix()
    rows.append(dict(directory=rel, n_files=len(files),
                     size_mb=round(sum(f.stat().st_size for f in files) / 1e6, 1),
                     first_file=files[0].name, modbus_in_probe=n_mb, top_pairs=top,
                     selected=next((s for s, v in CAPTURE_POINTS.items() if v == rel), '')))
survey = pd.DataFrame(rows)
survey.to_csv(WORK_DIR / 'capture_point_survey.csv', index=False)
print(f'Tìm thấy {len(survey)} thư mục chứa pcap')
pd.set_option('display.max_colwidth', 90)
survey

Khảo sát:   0%|          | 0/24 [00:00<?, ?it/s]

Tìm thấy 24 thư mục chứa pcap


,directory,n_files,size_mb,first_file,modbus_in_probe,top_pairs,selected
0,attack/compromised-ied/central-agent/central-agent-network-captures,5,481.7,vethffb308b-0.pcap,0,,
1,attack/compromised-ied/ied1a/ied1a-network-captures,6,541.9,veth3614724-0.pcap,17797,"SecHMI→IED1A:8899, IED1A→SecHMI:8898",
2,attack/compromised-ied/ied1b/ied1b-network-captures,6,520.0,vethc76bd3f-0.pcap,19862,"SecHMI→IED1B:9931, IED1B→SecHMI:9931",
3,attack/compromised-ied/ied4c/ied4c-network-captures,6,543.0,vethe685ac9-0.pcap,17768,"SecHMI→IED4C:8885, IED4C→SecHMI:8883",
4,attack/compromised-ied/trust-scada-hmi/trust-scada-network-captures,20,1995.4,veth3efd353-0.pcap,15857,"SecHMI→IED1B:2671, IED1B→SecHMI:2671, SecHMI→IED1A:2649",compromised_ied
5,attack/compromised-scada/central-agent/central-agent-network-captures,2,130.0,vethffb308b-0.pcap,0,,
6,attack/compromised-scada/ied1a/ied1a-network-captures,6,546.2,veth3614724-0.pcap,13623,"HMI→IED1A:9148, IED1A→HMI:4475",
7,attack/compromised-scada/ied1b/ied1b-network-captures,8,713.4,vethc76bd3f-0.pcap,21018,"IED1B→HMI:10511, HMI→IED1B:10507",
8,attack/compromised-scada/ied4c/ied4c-network-captures,6,535.4,vethe685ac9-0.pcap,13915,"HMI→IED4C:9901, IED4C→HMI:4014",
9,attack/compromised-scada/scada-hmi/scada-hmi-network-capture,17,1664.7,veth291ab9a-0.pcap,18683,"IED1B→HMI:3886, HMI→IED1B:3883, HMI→IED4C:2873",compromised_scada


Lấy danh sách các file có chứa gói Modbus

In [4]:
man = []
for scen, rel in CAPTURE_POINTS.items():
    files = sorted((DATA_ROOT / rel).glob('*.pcap'), key=natural_key)
    for i, f in enumerate(files):
        man.append(dict(scenario=scen, file_order=i, file=f.name, rel_path=f.relative_to(DATA_ROOT).as_posix(),
                        size_mb=round(f.stat().st_size / 1e6, 2)))
manifest = pd.DataFrame(man)
manifest['file_id'] = manifest['scenario'] + '/' + manifest['file'].str.replace('.pcap', '', regex=False)
manifest.to_csv(WORK_DIR / 'manifest.csv', index=False)

chk = manifest.groupby('scenario', sort=False).agg(n_files=('file', 'size'), size_mb=('size_mb', 'sum'))
chk['paper_files'] = chk.index.map(PAPER_FILES)
print('Tổng số file:', len(manifest))
chk

Tổng số file: 58


,n_files,size_mb,paper_files
scenario,,,
benign,19,1858.75,19
external,2,146.79,2
compromised_scada,17,1664.74,17
compromised_ied,20,1995.39,20


Trích xuất các trường dữ liệu (26 trường) bằng tshark
+ Thời gian, địa chỉ: frame.time_epoch, ip.src, ip.dst --> Gán nhãn theo cửa sổ thời gian, thống kê
+ IPv4 header (10 byte): ip.hdr_len, ip.dsfield, ip.len, ip.id, ip.flags, ip.frag_offset, ip.ttl, ip.proto 
+ TCP (8 byte): tcp.srcport, tcp.dstport, tcp.hdr_len, tcp.flags, tcp.window_size_value
+ MBAP + FC (8 byte): mbtcp.trans_id, mbtcp.prot_id, mbtcp.len, mbtcp.unit_id, modbus.func_code --> Dùng cho luật brute-force, length manipulation, frame stacking
+ PDU (4 byte): modbus.reference_num, modbus.word_cnt, modbus.bit_cnt, modbus.byte_cnt, tcp.payload --> Dùng cho luật False Data Injection

In [5]:
FIELDS = [
    ('frame.time_epoch', 'time'), ('ip.src', 'ip_src'), ('ip.dst', 'ip_dst'),
    ('ip.hdr_len', 'ip_hdr_len'), ('ip.dsfield', 'ip_dsfield'), ('ip.len', 'ip_len'), ('ip.id', 'ip_id'),
    ('ip.flags', 'ip_flags'), ('ip.frag_offset', 'ip_frag_offset'), ('ip.ttl', 'ip_ttl'), ('ip.proto', 'ip_proto'),
    ('tcp.srcport', 'tcp_srcport'), ('tcp.dstport', 'tcp_dstport'), ('tcp.hdr_len', 'tcp_hdr_len'),
    ('tcp.flags', 'tcp_flags'), ('tcp.window_size_value', 'tcp_window'),
    ('mbtcp.trans_id', 'mb_trans_id'), ('mbtcp.prot_id', 'mb_prot_id'), ('mbtcp.len', 'mb_len'),
    ('mbtcp.unit_id', 'mb_unit_id'), ('modbus.func_code', 'func_code_raw'),
    ('modbus.reference_num', 'reference_num'), ('modbus.word_cnt', 'word_cnt'), ('modbus.bit_cnt', 'bit_cnt'),
    ('modbus.byte_cnt', 'byte_cnt'), ('tcp.payload', 'payload'),
]
assert len(FIELDS) == 26
COLS = [c for _, c in FIELDS]
INT_COLS = ['ip_hdr_len', 'ip_dsfield', 'ip_len', 'ip_id', 'ip_flags', 'ip_frag_offset', 'ip_ttl', 'ip_proto',
            'tcp_srcport', 'tcp_dstport', 'tcp_hdr_len', 'tcp_flags', 'tcp_window',
            'mb_trans_id', 'mb_prot_id', 'mb_len', 'mb_unit_id', 'reference_num', 'word_cnt', 'bit_cnt', 'byte_cnt']

def to_int(s: pd.Series) -> pd.Series:
    """Chuỗi tshark -> int32. Lấy giá trị đầu tiên nếu có nhiều (a,b,c); hỗ trợ hex '0x..'; rỗng -> -1."""
    s = s.str.split(',', n=1).str[0].str.strip()
    out = pd.Series(-1, index=s.index, dtype='int64')
    is_hex = s.str.startswith('0x')
    if is_hex.any():
        out[is_hex] = s[is_hex].map(lambda x: int(x, 16))
    dec = (~is_hex) & (s != '')
    if dec.any():
        out[dec] = pd.to_numeric(s[dec], errors='coerce').fillna(-1).astype('int64')
    return out.astype('int32')

def parse_tshark_tsv(path: Path) -> pd.DataFrame:
    parts = []
    reader = pd.read_csv(path, sep='\t', header=None, names=COLS, dtype=str, quoting=3,
                         keep_default_na=False, na_filter=False, chunksize=500_000)
    for df in reader:
        out = pd.DataFrame({'time': pd.to_numeric(df['time'], errors='coerce')})
        out['ip_src'] = df['ip_src'].str.split(',', n=1).str[0]
        out['ip_dst'] = df['ip_dst'].str.split(',', n=1).str[0]
        for c in INT_COLS:
            out[c] = to_int(df[c])
        fc_raw = df['func_code_raw'].str.strip()
        out['func_code_raw'] = fc_raw
        out['func_code'] = to_int(fc_raw)
        out['n_pdus'] = np.where(fc_raw == '', 0, fc_raw.str.count(',') + 1).astype('int16')
        pay = df['payload'].str.replace(':', '', regex=False)
        out['payload_len'] = (pay.str.len() // 2).astype('int32')
        out['payload_hex12'] = pay.str[:24]
        parts.append(out)
    if not parts:
        return pd.DataFrame(columns=['time', 'ip_src', 'ip_dst'] + INT_COLS +
                            ['func_code_raw', 'func_code', 'n_pdus', 'payload_len', 'payload_hex12'])
    out = pd.concat(parts, ignore_index=True)
    out['ip_src'] = out['ip_src'].astype('category')
    out['ip_dst'] = out['ip_dst'].astype('category')
    return out

def extract_one(row) -> dict:
    src = DATA_ROOT / row.rel_path
    out_pq = WORK_DIR / 'extracted' / row.scenario / (Path(row.file).stem + '.parquet')
    if out_pq.exists():
        n = pd.read_parquet(out_pq, columns=['time'])
        return dict(file_id=row.file_id, n_packets=len(n), t_min=n.time.min() if len(n) else np.nan,
                    t_max=n.time.max() if len(n) else np.nan, seconds=0.0, status='cached')
    out_pq.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    pcap = src
    if LOCAL_CACHE is not None:
        LOCAL_CACHE.mkdir(parents=True, exist_ok=True)
        pcap = LOCAL_CACHE / f'{row.scenario}__{row.file}'
        shutil.copy(src, pcap)
    tsv = out_pq.with_suffix('.tsv.tmp') if LOCAL_CACHE is None else LOCAL_CACHE / (pcap.name + '.tsv')
    cmd = [TSHARK, '-n', '-r', str(pcap), '-Y', 'mbtcp', '-T', 'fields',
           '-E', 'header=n', '-E', 'separator=/t', '-E', 'occurrence=a', '-E', 'aggregator=,', '-E', 'quote=n']
    if not TCP_DESEGMENT:
        cmd += ['-o', 'tcp.desegment_tcp_streams:FALSE']
    if not TCP_ANALYZE_SEQ:
        cmd += ['-o', 'tcp.analyze_sequence_numbers:FALSE']
    for f, _ in FIELDS:
        cmd += ['-e', f]
    with open(tsv, 'w') as fo:
        p = subprocess.run(cmd, stdout=fo, stderr=subprocess.PIPE, text=True, encoding='utf-8', errors='replace')
    if p.returncode != 0:
        raise RuntimeError(f'tshark lỗi với {src}:\n{p.stderr[-1500:]}')
    df = parse_tshark_tsv(tsv)
    df.to_parquet(out_pq, index=False)
    tsv.unlink(missing_ok=True)
    if LOCAL_CACHE is not None:
        pcap.unlink(missing_ok=True)
    return dict(file_id=row.file_id, n_packets=len(df), t_min=df.time.min() if len(df) else np.nan,
                t_max=df.time.max() if len(df) else np.nan, seconds=round(time.time() - t0, 1), status='new')

Đọc attack log
Mỗi dòng log là một request tấn công, gồm 4 cột: Timestamp (UTC - ms), TargetIP, Attack, TransactionID

Chạy thử trên 1 file trước

In [6]:
smallest = manifest[manifest.size_mb > 0.1].sort_values('size_mb').iloc[0]
print('Chạy thử:', smallest.rel_path)
print(extract_one(smallest))
sample = pd.read_parquet(WORK_DIR / 'extracted' / smallest.scenario / (Path(smallest.file).stem + '.parquet'))
sample.head()

Chạy thử: attack/external/network-wide/network-wide-normal-1.pcap
{'file_id': 'external/network-wide-normal-1', 'n_packets': 74068, 't_min': np.float64(1675256800.878619), 't_max': np.float64(1675260110.737516), 'seconds': 13.9, 'status': 'new'}


,time,ip_src,ip_dst,ip_hdr_len,ip_dsfield,ip_len,ip_id,ip_flags,ip_frag_offset,ip_ttl,...,mb_unit_id,reference_num,word_cnt,bit_cnt,byte_cnt,func_code_raw,func_code,n_pdus,payload_len,payload_hex12
0,1.675257e+09,185.175.0.7,185.175.0.4,20,0,64,61045,2,0,64,...,1,0,-1,-1,-1,5,5,1,12,2b120000000601050000ff00
1,1.675257e+09,185.175.0.7,185.175.0.4,20,0,64,55117,2,0,64,...,1,0,-1,-1,-1,5,5,1,12,2b130000000601050000ff00
2,1.675257e+09,185.175.0.7,185.175.0.4,20,0,64,44797,2,0,64,...,1,0,-1,-1,-1,5,5,1,12,2b140000000601050000ff00
3,1.675257e+09,185.175.0.7,185.175.0.4,20,0,64,50299,2,0,64,...,1,0,-1,-1,-1,5,5,1,12,2b150000000601050000ff00
4,1.675257e+09,185.175.0.7,185.175.0.4,20,0,64,41336,2,0,64,...,1,0,-1,-1,-1,5,5,1,12,2b160000000601050000ff00


Chạy trên toàn bộ các file 

In [7]:
stats = []
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futs = {ex.submit(extract_one, r): r.file_id for r in manifest.itertuples()}
    for fu in tqdm(as_completed(futs), total=len(futs), desc='tshark'):
        stats.append(fu.result())
stats = pd.DataFrame(stats)
stats = manifest.merge(stats, on='file_id').sort_values(['scenario', 'file_order'])
stats['start_utc'] = pd.to_datetime(stats.t_min, unit='s')
stats['end_utc'] = pd.to_datetime(stats.t_max, unit='s')
stats.to_csv(WORK_DIR / 'extract_stats.csv', index=False)
print('Tổng thời gian tshark (file mới):', round(stats.seconds.sum() / 60, 1), 'phút')

tshark:   0%|          | 0/58 [00:00<?, ?it/s]

Tổng thời gian tshark (file mới): 54.8 phút


Đối chiếu với bài báo

In [8]:
summ = stats.groupby('scenario', sort=False).agg(
    files_with_modbus=('n_packets', lambda s: int((s > 0).sum())),
    n_packets=('n_packets', 'sum'),
    start_utc=('start_utc', 'min'), end_utc=('end_utc', 'max'))
summ['paper_files'] = summ.index.map(PAPER_FILES)
summ['paper_packets'] = summ.index.map(PAPER_PACKETS)
summ['diff_%'] = ((summ.n_packets / summ.paper_packets - 1) * 100).round(2)
tot = summ[['files_with_modbus', 'n_packets', 'paper_files', 'paper_packets']].sum()
print(f"Tổng: {int(tot.n_packets):,} gói từ {int(tot.files_with_modbus)} file "
      f"(bài báo: {int(tot.paper_packets):,} gói từ {int(tot.paper_files)} file)")
summ

Tổng: 11,390,143 gói từ 58 file (bài báo: 11,389,397 gói từ 58 file)


,files_with_modbus,n_packets,start_utc,end_utc,paper_files,paper_packets,diff_%
scenario,,,,,,,
benign,19,4101460,2023-01-23 13:52:04.004200935,2023-01-31 16:25:58.037503958,19,4101373,0.00
compromised_ied,20,3348713,2023-03-23 02:31:40.940954924,2023-03-29 16:15:45.449326038,20,3348642,0.00
compromised_scada,17,3657891,2023-03-15 02:01:08.009526014,2023-03-22 03:55:20.413600922,17,3657304,0.02
external,2,282079,2023-02-01 03:48:39.921816111,2023-02-01 14:01:50.737515926,2,282078,0.00


In [9]:
# Chi tiết từng file (file không có Modbus, ví dụ substation-0, sẽ hiện n_packets = 0)
stats[['scenario', 'file', 'size_mb', 'n_packets', 'start_utc', 'end_utc', 'status']]

,scenario,file,size_mb,n_packets,start_utc,end_utc,status
0,benign,veth5bbeaa2-normal-13.pcap,100.00,220646,2023-01-23 13:52:04.004200935,2023-01-24 00:36:26.656312943,new
1,benign,veth5bbeaa2-normal-14.pcap,100.00,220661,2023-01-24 00:36:30.864227057,2023-01-24 11:22:07.276561975,new
2,benign,veth5bbeaa2-normal-15.pcap,100.00,220644,2023-01-24 11:22:07.297564030,2023-01-24 21:51:34.576545000,new
3,benign,veth5bbeaa2-normal-16.pcap,100.00,220719,2023-01-24 21:51:34.597103119,2023-01-25 08:18:57.120616913,new
4,benign,veth5bbeaa2-normal-17.pcap,100.00,220679,2023-01-25 08:18:57.121849060,2023-01-25 18:45:58.596499920,new
5,benign,veth5bbeaa2-normal-18.pcap,100.00,220600,2023-01-25 18:45:58.617347002,2023-01-26 05:11:54.528213024,new
6,benign,veth5bbeaa2-normal-19.pcap,100.00,220683,2023-01-26 05:11:57.932204962,2023-01-26 15:38:22.917893887,new
7,benign,veth5bbeaa2-normal-20.pcap,100.00,220681,2023-01-26 15:38:22.930305004,2023-01-27 02:09:46.985690117,new
8,benign,veth5bbeaa2-normal-21.pcap,100.00,220714,2023-01-27 02:09:46.985866070,2023-01-27 12:35:25.880033016,new
9,benign,veth5bbeaa2-normal-22.pcap,100.00,220731,2023-01-27 12:35:25.886523962,2023-01-27 23:03:52.612628937,new


In [10]:
def load_scenario(scen, columns=None):
    files = sorted((WORK_DIR / 'extracted' / scen).glob('*.parquet'), key=natural_key)
    dfs = [pd.read_parquet(f, columns=columns) for f in files]
    dfs = [d for d in dfs if len(d)]
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(columns=columns)

cols = ['func_code', 'n_pdus', 'byte_cnt', 'mb_len']
rows = []
for scen in CAPTURE_POINTS:
    d = load_scenario(scen, cols)
    vc = d.func_code.value_counts()
    rows.append(dict(scenario=scen, n=len(d),
                     **{f'FC{k}': int(vc.get(k, 0)) for k in [1, 2, 3, 4, 5, 6, 15, 16]},
                     FC_other=int(len(d) - vc.reindex([1, 2, 3, 4, 5, 6, 15, 16]).fillna(0).sum()),
                     stacked_npdus_gt1=int((d.n_pdus > 1).sum()),
                     byte_cnt_171=int((d.byte_cnt == 171).sum())))
pd.DataFrame(rows).set_index('scenario')

,n,FC1,FC2,FC3,FC4,FC5,FC6,FC15,FC16,FC_other,stacked_npdus_gt1,byte_cnt_171
scenario,,,,,,,,,,,,
benign,4101460,1627092,813530,831684,813604,4150,11400,0,0,0,0,0
external,282079,86347,43193,43202,43197,65804,336,0,0,0,1,0
compromised_scada,3657891,1152912,577829,621012,578571,718317,9250,0,0,0,1318,0
compromised_ied,3348713,1334178,666925,666568,666570,3226,11246,0,0,0,127,53
